Import Libraries

In [5]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import (
    pad_sequence,
    pack_padded_sequence,
    pad_packed_sequence
)
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, roc_auc_score
import numpy as np
import pandas as pd
import pickle
from collections import defaultdict


In [6]:
def recall_at_k(y_true, y_score, k=10):
    """
    Compute Recall@K for multi-label drug recommendation.
    y_true: (N, D) binary multi-hot
    y_score: (N, >=D) predicted scores (may include padding)
    """
    recalls = []
    for yt, ys in zip(y_true, y_score):
        D = yt.shape[0]          # true number of drugs
        ys = ys[:D]              # 🔑 truncate padded scores

        topk_idx = np.argsort(ys)[::-1][:min(k, D)]
        hits = yt[topk_idx].sum()
        recall = hits / yt.sum() if yt.sum() > 0 else 0.0
        recalls.append(recall)

    return np.array(recalls)


In [7]:
def ndcg_at_k(y_true, y_score, k=10):
    """
    NDCG@K for multi-label recommendation.
    y_true: (N, D) multi-hot
    y_score: (N, D) predicted scores
    """
    ndcgs = []

    for yt, ys in zip(y_true, y_score):
        D = yt.shape[0]
        ys = ys[:D]

        k_eff = min(k, D)

        # Top-k indices
        ranked_idx = np.argsort(-ys)[:k_eff]

        # DCG
        gains = yt[ranked_idx]
        discounts = 1.0 / np.log2(np.arange(2, k_eff + 2))
        dcg = np.sum(gains * discounts)

        # IDCG (ideal ranking)
        ideal_gains = np.sort(yt)[::-1][:k_eff]
        idcg = np.sum(ideal_gains * discounts)

        if idcg == 0:
            ndcgs.append(0.0)
        else:
            ndcgs.append(dcg / idcg)

    return np.mean(ndcgs)


In [8]:
def ddi_rate (y_score, ddi_matrix, top_k=10):
    """
    Recommended-set DDI@K 

    y_score: (N, D) predicted scores
    ddi_matrix: (D, D) severity or binary DDI matrix
    """
    ddi_count = 0
    pair_count = 0
    severity_sum = 0.0

    for ys in y_score:
        D = ddi_matrix.shape[0]
        ys = ys[:D]  # truncate padding if any

        topk_idx = np.argsort(ys)[::-1][:min(top_k, D)]

        for i, d1 in enumerate(topk_idx):
            for d2 in topk_idx[i + 1:]:
                pair_count += 1
                sev = ddi_matrix[d1, d2]
                if sev > 0:
                    ddi_count += 1
                    severity_sum += sev

    ddi_rate_val = ddi_count / pair_count if pair_count > 0 else 0.0
    avg_severity = severity_sum / ddi_count if ddi_count > 0 else 0.0

    return ddi_rate_val, avg_severity


Data Preprocessing

In [9]:
# ============================================================
#  COMPLETE PIPELINE: EXPANDED CANCER COVERAGE
# WITHOUT CLASS WEIGHTS
# ============================================================

import numpy as np
import pandas as pd
import pickle
from collections import defaultdict
from sklearn.preprocessing import LabelEncoder

print("\n" + "="*70)
print("EXPANDED CANCER CLASSIFICATION + ABLATION STUDY")
print("="*70)

# =========================================================
# ABLATION STUDY CONFIGURATION
# =========================================================
USE_RANDOM_LAB = False       
USE_RANDOM_RAD = False       
USE_RANDOM_DRUG_DESC = False  
USE_HIERARCHICAL = True       
RANDOM_SEED = 42

# =========================================================
# PATHS
# =========================================================
EMB_RAD_PATH = r"....cancer_admission_embs_radiology.npy"
EMB_LAB_PATH = r"....solid_cancer_lab_embs.npy"
DRUG_EMB_PATH = r"......cancer_admission_embs_drugs.npy"
DRUG_SEQ_PATH = r".....cancer_drug_sequences.npy"
DDI_PATH = r".....mapped_ddi_pairs.pkl"
DIAGNOSES_PATH = r"......mimic iv\mimic-iv-3.1\hosp\diagnoses_icd.csv.gz"
DRUG2IDX_PATH = r".....drug2idx.pkl"

# =========================================================
# STEP 1: EXPANDED ICD CANCER CODE LISTS
# =========================================================
print("\n" + "-"*40)
print("DEFINING EXPANDED CANCER TYPE MAPPING...")

# Complete ICD-9 Cancer Codes (ALL solid tumors)
EXPANDED_ICD9 = (
    # Head & Neck (140-149)
    "140","141","142","143","144","145","146","147","148","149",
    # Digestive (150-159)
    "150","151","152","153","154","155","156","157","158","159",
    # Respiratory (160-165)
    "160","161","162","163","164","165",
    # Bone/Soft Tissue (170-176)
    "170","171","172","173","174","175","176",
    # Genitourinary (179-189)
    "179","180","181","182","183","184","185","186","187","188","189",
    # Brain/CNS (190-192)
    "190","191","192",
    # Thyroid/Endocrine (193-199)
    "193","194","195","196","197","198","199"
)

# Complete ICD-10 Cancer Codes (ALL solid tumors)
EXPANDED_ICD10 = (
    # Head & Neck (C00-C14)
    "C00","C01","C02","C03","C04","C05","C06","C07","C08","C09","C10","C11","C12","C13","C14",
    # Digestive (C15-C26)
    "C15","C16","C17","C18","C19","C20","C21","C22","C23","C24","C25","C26",
    # Respiratory (C30-C39)
    "C30","C31","C32","C33","C34","C37","C38","C39",
    # Bone/Soft Tissue (C40-C49)
    "C40","C41","C43","C44","C45","C46","C47","C48","C49",
    # Breast (C50)
    "C50",
    # Female Genital (C51-C58)
    "C51","C52","C53","C54","C55","C56","C57","C58",
    # Male Genital (C60-C63)
    "C60","C61","C62","C63",
    # Urinary (C64-C68)
    "C64","C65","C66","C67","C68",
    # Brain/CNS (C69-C72)
    "C69","C70","C71","C72",
    # Thyroid/Endocrine (C73-C75)
    "C73","C74","C75"
)

print(f" Expanded ICD-9 prefixes: {len(EXPANDED_ICD9)}")
print(f" Expanded ICD-10 prefixes: {len(EXPANDED_ICD10)}")

# =========================================================
# STEP 2: COMPREHENSIVE ICD TO CANCER TYPE MAPPING
# =========================================================

# Complete ICD-9 to Cancer Type Mapping
ICD9_CANCER_MAPPING = {
    # Head and Neck (140-149)
    '140': 'HEAD_NECK_CANCER', '141': 'HEAD_NECK_CANCER', '142': 'HEAD_NECK_CANCER',
    '143': 'HEAD_NECK_CANCER', '144': 'HEAD_NECK_CANCER', '145': 'HEAD_NECK_CANCER',
    '146': 'HEAD_NECK_CANCER', '147': 'HEAD_NECK_CANCER', '148': 'HEAD_NECK_CANCER',
    '149': 'HEAD_NECK_CANCER',
    
    # Digestive System
    '150': 'ESOPHAGEAL_CANCER', '151': 'STOMACH_CANCER',
    '152': 'SMALL_INTESTINE_CANCER', '153': 'COLORECTAL_CANCER', '154': 'COLORECTAL_CANCER',
    '155': 'LIVER_CANCER', '156': 'GALLBLADDER_CANCER', '157': 'PANCREATIC_CANCER',
    '158': 'PERITONEAL_CANCER', '159': 'OTHER_DIGESTIVE_CANCER',
    
    # Respiratory
    '160': 'NASAL_CANCER', '161': 'LARYNGEAL_CANCER',
    '162': 'LUNG_CANCER', '163': 'PLEURAL_CANCER', '164': 'THYMUS_CANCER', 
    '165': 'OTHER_RESPIRATORY_CANCER',
    
    # Bone and Soft Tissue
    '170': 'BONE_CANCER', '171': 'SOFT_TISSUE_CANCER', '172': 'MELANOMA',
    '173': 'OTHER_SKIN_CANCER', '174': 'BREAST_CANCER', '175': 'MALE_BREAST_CANCER',
    '176': 'KAPOSI_SARCOMA',
    
    # Genitourinary
    '179': 'UTERINE_CANCER', '180': 'CERVICAL_CANCER', '181': 'PLACENTAL_CANCER',
    '182': 'OVARIAN_CANCER', '183': 'OTHER_FEMALE_GENITAL', '184': 'VULVAR_CANCER',
    '185': 'PROSTATE_CANCER', '186': 'TESTICULAR_CANCER', '187': 'PENILE_CANCER',
    '188': 'BLADDER_CANCER', '189': 'KIDNEY_CANCER',
    
    # Brain and CNS
    '190': 'EYE_CANCER', '191': 'BRAIN_CANCER', '192': 'SPINAL_CORD_CANCER',
    
    # Thyroid and Endocrine
    '193': 'THYROID_CANCER', '194': 'ENDOCRINE_CANCER',
    '195': 'OTHER_CANCER', '196': 'METASTATIC_CANCER', '197': 'SECONDARY_RESPIRATORY',
    '198': 'SECONDARY_DIGESTIVE', '199': 'SECONDARY_CANCER',
}

# Complete ICD-10 to Cancer Type Mapping
ICD10_CANCER_MAPPING = {
    # Head and Neck
    'C00': 'HEAD_NECK_CANCER', 'C01': 'HEAD_NECK_CANCER', 'C02': 'HEAD_NECK_CANCER',
    'C03': 'HEAD_NECK_CANCER', 'C04': 'HEAD_NECK_CANCER', 'C05': 'HEAD_NECK_CANCER',
    'C06': 'HEAD_NECK_CANCER', 'C07': 'HEAD_NECK_CANCER', 'C08': 'HEAD_NECK_CANCER',
    'C09': 'HEAD_NECK_CANCER', 'C10': 'HEAD_NECK_CANCER', 'C11': 'HEAD_NECK_CANCER',
    'C12': 'HEAD_NECK_CANCER', 'C13': 'HEAD_NECK_CANCER', 'C14': 'HEAD_NECK_CANCER',
    
    # Digestive
    'C15': 'ESOPHAGEAL_CANCER', 'C16': 'STOMACH_CANCER', 'C17': 'SMALL_INTESTINE_CANCER',
    'C18': 'COLORECTAL_CANCER', 'C19': 'COLORECTAL_CANCER', 'C20': 'COLORECTAL_CANCER',
    'C21': 'ANAL_CANCER', 'C22': 'LIVER_CANCER', 'C23': 'GALLBLADDER_CANCER',
    'C24': 'BILE_DUCT_CANCER', 'C25': 'PANCREATIC_CANCER', 'C26': 'OTHER_DIGESTIVE_CANCER',
    
    # Respiratory
    'C30': 'NASAL_CANCER', 'C31': 'SINUS_CANCER', 'C32': 'LARYNGEAL_CANCER',
    'C33': 'TRACHEAL_CANCER', 'C34': 'LUNG_CANCER', 'C37': 'THYMUS_CANCER',
    'C38': 'HEART_MEDIASTINAL_CANCER', 'C39': 'OTHER_RESPIRATORY_CANCER',
    
    # Bone and Soft Tissue
    'C40': 'BONE_CANCER', 'C41': 'BONE_CANCER', 'C43': 'MELANOMA',
    'C44': 'OTHER_SKIN_CANCER', 'C45': 'MESOTHELIOMA', 'C46': 'KAPOSI_SARCOMA',
    'C47': 'PERIPHERAL_NERVE_CANCER', 'C48': 'RETROPERITONEAL_CANCER', 'C49': 'SOFT_TISSUE_CANCER',
    
    # Breast
    'C50': 'BREAST_CANCER',
    
    # Female Genital
    'C51': 'VULVAR_CANCER', 'C52': 'VAGINAL_CANCER', 'C53': 'CERVICAL_CANCER',
    'C54': 'ENDOMETRIAL_CANCER', 'C55': 'UTERINE_CANCER', 'C56': 'OVARIAN_CANCER',
    'C57': 'OTHER_FEMALE_GENITAL', 'C58': 'PLACENTAL_CANCER',
    
    # Male Genital
    'C60': 'PENILE_CANCER', 'C61': 'PROSTATE_CANCER', 'C62': 'TESTICULAR_CANCER',
    'C63': 'OTHER_MALE_GENITAL',
    
    # Urinary
    'C64': 'KIDNEY_CANCER', 'C65': 'RENAL_PELVIS_CANCER', 'C66': 'URETERAL_CANCER',
    'C67': 'BLADDER_CANCER', 'C68': 'OTHER_URINARY_CANCER',
    
    # Brain and CNS
    'C69': 'EYE_CANCER', 'C70': 'MENINGEAL_CANCER', 'C71': 'BRAIN_CANCER',
    'C72': 'SPINAL_CORD_CANCER',
    
    # Thyroid and Endocrine
    'C73': 'THYROID_CANCER', 'C74': 'ADRENAL_CANCER', 'C75': 'OTHER_ENDOCRINE_CANCER',
}

print(f" ICD-9 mapped: {len(ICD9_CANCER_MAPPING)} codes")
print(f" ICD-10 mapped: {len(ICD10_CANCER_MAPPING)} codes")

# =========================================================
# STEP 3: FUNCTION TO MAP ICD TO CANCER TYPE
# =========================================================

def map_icd_to_cancer_type(icd_code, icd_version):
    """Map ICD code to specific cancer type using comprehensive mappings"""
    icd_code = str(icd_code).upper().strip()
    
    if icd_version == 9:
        for code_prefix, cancer_type in ICD9_CANCER_MAPPING.items():
            if icd_code.startswith(code_prefix):
                return cancer_type
    elif icd_version == 10:
        for code_prefix, cancer_type in ICD10_CANCER_MAPPING.items():
            if icd_code.startswith(code_prefix):
                return cancer_type
    
    return 'OTHER_CANCER'

# =========================================================
# STEP 4: HIERARCHICAL COARSE CLASS MAPPING
# =========================================================

# Group similar cancer types for balanced classification
COARSE_CANCER_GROUPS = {
    'LUNG_CANCER': ['LUNG_CANCER', 'TRACHEAL_CANCER'],
    'BREAST_CANCER': ['BREAST_CANCER', 'MALE_BREAST_CANCER'],
    'COLORECTAL_CANCER': ['COLORECTAL_CANCER', 'ANAL_CANCER'],
    'PROSTATE_CANCER': ['PROSTATE_CANCER'],
    'BLADDER_CANCER': ['BLADDER_CANCER'],
    'KIDNEY_CANCER': ['KIDNEY_CANCER', 'RENAL_PELVIS_CANCER'],
    'STOMACH_CANCER': ['STOMACH_CANCER'],
    'LIVER_CANCER': ['LIVER_CANCER', 'BILE_DUCT_CANCER'],
    'PANCREATIC_CANCER': ['PANCREATIC_CANCER'],
    'ESOPHAGEAL_CANCER': ['ESOPHAGEAL_CANCER'],
    'OVARIAN_CANCER': ['OVARIAN_CANCER'],
    'CERVICAL_CANCER': ['CERVICAL_CANCER'],
    'UTERINE_CANCER': ['UTERINE_CANCER', 'ENDOMETRIAL_CANCER'],
    'HEAD_NECK_CANCER': ['HEAD_NECK_CANCER', 'LARYNGEAL_CANCER', 'NASAL_CANCER', 
                         'SINUS_CANCER', 'ORAL_CANCER', 'SALIVARY_CANCER'],
    'THYROID_CANCER': ['THYROID_CANCER'],
    'BRAIN_CANCER': ['BRAIN_CANCER', 'SPINAL_CORD_CANCER', 'MENINGEAL_CANCER'],
    'MELANOMA': ['MELANOMA', 'OTHER_SKIN_CANCER'],
    'KAPOSI_SARCOMA': ['KAPOSI_SARCOMA', 'MESOTHELIOMA'],
    'OTHER_CANCER': ['OTHER_CANCER', 'METASTATIC_CANCER', 'SECONDARY_CANCER',
                     'OTHER_DIGESTIVE_CANCER', 'OTHER_RESPIRATORY_CANCER',
                     'OTHER_FEMALE_GENITAL', 'OTHER_MALE_GENITAL', 'OTHER_URINARY_CANCER',
                     'OTHER_ENDOCRINE_CANCER', 'PERITONEAL_CANCER', 'PLEURAL_CANCER',
                     'THYMUS_CANCER', 'HEART_MEDIASTINAL_CANCER', 'RETROPERITONEAL_CANCER',
                     'PERIPHERAL_NERVE_CANCER', 'SOFT_TISSUE_CANCER', 'BONE_CANCER',
                     'EYE_CANCER', 'ADRENAL_CANCER', 'TESTICULAR_CANCER', 'PENILE_CANCER',
                     'VULVAR_CANCER', 'VAGINAL_CANCER', 'PLACENTAL_CANCER', 'GALLBLADDER_CANCER',
                     'SMALL_INTESTINE_CANCER', 'URETERAL_CANCER']
}

def map_to_coarse_class(cancer_type):
    """Map fine-grained cancer type to coarse category"""
    for coarse_group, fine_types in COARSE_CANCER_GROUPS.items():
        if cancer_type in fine_types:
            return coarse_group
    return 'OTHER_CANCER'

# =========================================================
# STEP 5: HELPER FUNCTION FOR RANDOM EMBEDDINGS
# =========================================================
def maybe_replace_with_random(embeddings, use_random, modality_name, seed=None):
    if not use_random:
        print(f"✓ Using REAL {modality_name} embeddings")
        return embeddings
    
    print(f" Using RANDOM {modality_name} embeddings (seed={seed})")
    if seed is not None:
        np.random.seed(seed)
    
    mean = np.mean(embeddings)
    std = np.std(embeddings)
    random_embs = np.random.normal(mean, std, size=embeddings.shape)
    
    original_norms = np.linalg.norm(embeddings, axis=1)
    random_norms = np.linalg.norm(random_embs, axis=1)
    scale_factors = original_norms / (random_norms + 1e-8)
    random_embs = random_embs * scale_factors[:, np.newaxis]
    
    return random_embs.astype(embeddings.dtype)

# =========================================================
# STEP 6: LOAD AND PROCESS EMBEDDINGS
# =========================================================
print("\n" + "="*60)
print("LOADING AND PROCESSING EMBEDDINGS")
print("="*60)

X_rad = np.load(EMB_RAD_PATH)
hadm_ids_rad = np.load(EMB_RAD_PATH.replace(".npy","_hadm_ids.npy"))
subject_ids_rad = np.load(EMB_RAD_PATH.replace(".npy","_subject_ids.npy"))

X_lab = np.load(EMB_LAB_PATH)
hadm_ids_lab = np.load(EMB_LAB_PATH.replace(".npy","_hadm_ids.npy"))
subject_ids_lab = np.load(EMB_LAB_PATH.replace(".npy","_subject_ids.npy"))

print(f"\nOriginal shapes:")
print(f"  Radiology: {X_rad.shape}")
print(f"  Lab: {X_lab.shape}")

# Apply random embeddings
X_lab = maybe_replace_with_random(X_lab, USE_RANDOM_LAB, "LAB", RANDOM_SEED)
X_rad = maybe_replace_with_random(X_rad, USE_RANDOM_RAD, "RADIOLOGY", RANDOM_SEED)

# Align admissions
common_hadm_ids = np.intersect1d(hadm_ids_rad, hadm_ids_lab)
idx_rad = np.isin(hadm_ids_rad, common_hadm_ids)
idx_lab = np.isin(hadm_ids_lab, common_hadm_ids)

X_rad = X_rad[idx_rad]
X_lab = X_lab[idx_lab]
hadm_ids = hadm_ids_lab[idx_lab]
subject_ids = subject_ids_lab[idx_lab]

print(f"\nAfter alignment:")
print(f"  Radiology: {X_rad.shape}")
print(f"  Lab: {X_lab.shape}")

# =========================================================
# STEP 7: LOAD DRUG DATA
# =========================================================
print("\n" + "-"*40)
print("LOADING DRUG DATA...")

X_drug = np.load(DRUG_EMB_PATH)
hadm_ids_drug = np.load(DRUG_EMB_PATH.replace(".npy","_hadm_ids.npy"))
drug_sequences = np.load(DRUG_SEQ_PATH)
drug_lengths = np.load(DRUG_SEQ_PATH.replace(".npy","_lengths.npy"))

print(f"Drug embeddings shape: {X_drug.shape}")
print(f"Drug sequences shape: {drug_sequences.shape}")

# Apply random embeddings to drug descriptions
X_drug = maybe_replace_with_random(X_drug, USE_RANDOM_DRUG_DESC, "DRUG DESCRIPTION", RANDOM_SEED)

# =========================================================
# STEP 8: ALIGN ALL DATA
# =========================================================
print("\n" + "-"*40)
print("ALIGNING DATA...")

drug_idx = {hid: i for i, hid in enumerate(hadm_ids_drug)}

aligned_X_drug = []
aligned_drug_sequences = []
aligned_drug_lengths = []
aligned_X_lab = []
aligned_X_rad = []
aligned_subject_ids = []
aligned_hadm_ids = []
missing_drug = 0

for i, hid in enumerate(hadm_ids):
    if hid in drug_idx:
        j = drug_idx[hid]
        aligned_X_drug.append(X_drug[j])
        aligned_drug_sequences.append(drug_sequences[j])
        aligned_drug_lengths.append(drug_lengths[j])
        aligned_X_lab.append(X_lab[i])
        aligned_X_rad.append(X_rad[i])
        aligned_subject_ids.append(subject_ids[i])
        aligned_hadm_ids.append(hid)
    else:
        missing_drug += 1

print(f"Admissions missing drug data: {missing_drug}")

# Stack arrays
X_drug = np.stack(aligned_X_drug)
drug_sequences = np.stack(aligned_drug_sequences)
drug_lengths = np.array(aligned_drug_lengths)
X_lab = np.stack(aligned_X_lab)
X_rad = np.stack(aligned_X_rad)
subject_ids = np.array(aligned_subject_ids)
hadm_ids = np.array(aligned_hadm_ids)

print(f"\nAligned shapes:")
print(f"  Lab: {X_lab.shape}")
print(f"  Rad: {X_rad.shape}")
print(f"  Drug desc: {X_drug.shape}")
print(f"  Drug seq: {drug_sequences.shape}")

# =========================================================
# STEP 9: LOAD AND MAP DIAGNOSIS LABELS (UPDATED)
# =========================================================
print("\n" + "-"*40)
print("LOADING DIAGNOSIS LABELS...")

diag = pd.read_csv(
    DIAGNOSES_PATH,
    usecols=["hadm_id","icd_code","icd_version","seq_num"]
)
diag["icd_code"] = diag["icd_code"].astype(str).str.upper().str.strip()

# Use EXPANDED cancer filters
mask = (
    ((diag.icd_version==9) & diag.icd_code.str.startswith(EXPANDED_ICD9)) |
    ((diag.icd_version==10) & diag.icd_code.str.startswith(EXPANDED_ICD10))
)

hf_diag = diag[mask]
hadm_to_cancer_type = {}

for hid, g in hf_diag.groupby("hadm_id"):
    primary = g[g.seq_num.isin([1,2])]
    if len(primary) > 0:
        row = primary.sort_values("seq_num").iloc[0]
        cancer_type = map_icd_to_cancer_type(row["icd_code"], row["icd_version"])
        hadm_to_cancer_type[hid] = cancer_type
    else:
        # If no primary, take most common cancer type in admission
        most_common = g['icd_code'].mode()
        if len(most_common) > 0:
            cancer_type = map_icd_to_cancer_type(most_common[0], g.iloc[0]['icd_version'])
            hadm_to_cancer_type[hid] = cancer_type
        else:
            hadm_to_cancer_type[hid] = "OTHER_CANCER"

# Apply diagnosis filter
final_mask = np.array([hid in hadm_to_cancer_type for hid in hadm_ids])
X_lab = X_lab[final_mask]
X_rad = X_rad[final_mask]
X_drug = X_drug[final_mask]
drug_sequences = drug_sequences[final_mask]
drug_lengths = drug_lengths[final_mask]
subject_ids = subject_ids[final_mask]
hadm_ids = hadm_ids[final_mask]

y_hf_fine = np.array([hadm_to_cancer_type[hid] for hid in hadm_ids])

print(f"After diagnosis filter: {len(y_hf_fine)} admissions")

# =========================================================
# STEP 10: COLLAPSE VERY RARE CLASSES
# =========================================================
print("\n" + "-"*40)
print("COLLAPSING VERY RARE CLASSES...")

MIN_SAMPLES = 100
counts = pd.Series(y_hf_fine).value_counts()
rare = counts[counts < MIN_SAMPLES].index

y_hf_fine = np.array([
    "RARE_CANCER" if lbl in rare else lbl
    for lbl in y_hf_fine
])

print("\nFine-grained label distribution:")
label_counts = pd.Series(y_hf_fine).value_counts()
for label, count in label_counts.items():
    pct = count / len(y_hf_fine) * 100
    bar = '' * int(pct / 2)
    print(f"  {label:30s}: {count:5d} ({pct:5.1f}%) {bar}")

# =========================================================
# STEP 11: APPLY HIERARCHICAL MAPPING
# =========================================================
print("\n" + "-"*40)
print("APPLYING HIERARCHICAL MAPPING...")

if USE_HIERARCHICAL:
    y_hf = np.array([map_to_coarse_class(lbl) for lbl in y_hf_fine])
    print("\n Using HIERARCHICAL coarse classes:")
else:
    y_hf = y_hf_fine.copy()
    print("\n Using FINE-grained original classes:")

label_counts = pd.Series(y_hf).value_counts()
total = len(y_hf)
print(f"\nFinal label distribution ({len(label_counts)} classes):")
for label, count in label_counts.items():
    pct = count / total * 100
    bar = '' * int(pct / 2)
    print(f"  {label:30s}: {count:5d} ({pct:5.1f}%) {bar}")

# =========================================================
# STEP 12: AGGREGATE BY PATIENT
# =========================================================
print("\n" + "-"*40)
print("AGGREGATING BY PATIENT...")

patient_to_lab = defaultdict(list)
patient_to_rad = defaultdict(list)
patient_to_drug_desc = defaultdict(list)
patient_to_labels = defaultdict(list)
patient_to_drugs = defaultdict(list)

for lab, rad, drug_desc, lbl, drug_seq, pid in zip(
    X_lab, X_rad, X_drug, y_hf, drug_sequences, subject_ids
):
    patient_to_lab[pid].append(lab)
    patient_to_rad[pid].append(rad)
    patient_to_drug_desc[pid].append(drug_desc)
    patient_to_labels[pid].append(lbl)
    patient_to_drugs[pid].append(drug_seq)

lab_sequences_by_patient = [np.stack(v) for v in patient_to_lab.values()]
rad_sequences_by_patient = [np.stack(v) for v in patient_to_rad.values()]
drug_desc_sequences_by_patient = [np.stack(v) for v in patient_to_drug_desc.values()]
drug_labels_by_patient = [np.stack(v) for v in patient_to_drugs.values()]
labels_by_patient = [np.array(v) for v in patient_to_labels.values()]
patient_ids = list(patient_to_lab.keys())

print(f"\nFinal dataset:")
print(f"  Patients: {len(patient_ids)}")
print(f"  Total admissions: {sum(len(seq) for seq in labels_by_patient)}")

# =========================================================
# STEP 13: LOAD DDI MATRIX
# =========================================================
print("\n" + "-"*40)
print("LOADING DDI MATRIX...")

with open(DDI_PATH, "rb") as f:
    mapped_ddi_pairs = pickle.load(f)

with open(DRUG2IDX_PATH, "rb") as f:
    drug2idx = pickle.load(f)

def normalize_drug_name(name):
    if name is None:
        return ""
    name = name.lower().strip()
    if name.startswith("*nf*"):
        name = name[4:].strip()
    return name

norm_drug2idx = {normalize_drug_name(d): idx for d, idx in drug2idx.items()}
n_drugs = len(norm_drug2idx)
print(f"Number of drugs: {n_drugs}")

severity_weight = {"minor": 0.5, "moderate": 2.0, "major": 5.0}

def build_ddi_severity_matrices(ddi_pairs, drug2idx, n_drugs, severity_weight):
    ddi_matrix = np.zeros((n_drugs, n_drugs), dtype=np.float32)
    for drug1, drug2, severity in ddi_pairs:
        if drug1 in drug2idx and drug2 in drug2idx:
            i, j = drug2idx[drug1], drug2idx[drug2]
            weight = severity_weight.get(severity, 1.0)
            ddi_matrix[i, j] = weight
            ddi_matrix[j, i] = weight
    return ddi_matrix

ddi_severity_matrix = build_ddi_severity_matrices(
    ddi_pairs=mapped_ddi_pairs,
    drug2idx=norm_drug2idx,
    n_drugs=n_drugs,
    severity_weight=severity_weight
)

print(f"DDI severity matrix shape: {ddi_severity_matrix.shape}")

# =========================================================
# STEP 14: CREATE LABEL ENCODER FOR TRAINING
# =========================================================
print("\n" + "-"*40)
print("CREATING LABEL ENCODER...")

le_hf = LabelEncoder()
all_labels = np.concatenate(labels_by_patient)
le_hf.fit(all_labels)
labels_by_patient_enc = [le_hf.transform(seq) for seq in labels_by_patient]
n_classes = len(le_hf.classes_)

print(f"Number of classes for training: {n_classes}")
print(f"Classes: {list(le_hf.classes_)}")

# =========================================================
# STEP 15: FINAL VALIDATION
# =========================================================
print("\n" + "="*60)
print("VALIDATION")
print("="*60)

n = len(hadm_ids)
assert X_lab.shape[0] == n, f"Lab shape mismatch"
assert X_rad.shape[0] == n, f"Rad shape mismatch"
assert X_drug.shape[0] == n, f"Drug desc shape mismatch"

print("\n PREPROCESSING COMPLETE!")
print(f"\n Ablation configuration:")
print(f"   - LAB: {'RANDOM' if USE_RANDOM_LAB else 'REAL'}")
print(f"   - RADIOLOGY: {'RANDOM' if USE_RANDOM_RAD else 'REAL'}")
print(f"   - DRUG DESCRIPTION: {'RANDOM' if USE_RANDOM_DRUG_DESC else 'REAL'}")
print(f"   - HIERARCHICAL CLASSES: {'YES' if USE_HIERARCHICAL else 'NO'}")

print(f"\n Dataset statistics:")
print(f"   - Patients: {len(patient_ids)}")
print(f"   - Admissions: {n}")
print(f"   - Classes: {n_classes}")
print(f"   - Drugs: {n_drugs}")

print(f"\n Final class distribution:")
for cls, count in label_counts.items():
    pct = count / total * 100
    bar = '' * int(pct / 2)
    print(f"  {cls:30s}: {count:5d} ({pct:5.1f}%) {bar}")


EXPANDED CANCER CLASSIFICATION + ABLATION STUDY

----------------------------------------
DEFINING EXPANDED CANCER TYPE MAPPING...
✅ Expanded ICD-9 prefixes: 54
✅ Expanded ICD-10 prefixes: 69
✅ ICD-9 mapped: 54 codes
✅ ICD-10 mapped: 69 codes

LOADING AND PROCESSING EMBEDDINGS

Original shapes:
  Radiology: (12708, 2560)
  Lab: (19866, 2560)
✓ Using REAL LAB embeddings
✓ Using REAL RADIOLOGY embeddings

After alignment:
  Radiology: (12093, 2560)
  Lab: (12093, 2560)

----------------------------------------
LOADING DRUG DATA...
Drug embeddings shape: (21158, 2560)
Drug sequences shape: (21173, 689)
✓ Using REAL DRUG DESCRIPTION embeddings

----------------------------------------
ALIGNING DATA...
Admissions missing drug data: 41

Aligned shapes:
  Lab: (12052, 2560)
  Rad: (12052, 2560)
  Drug desc: (12052, 2560)
  Drug seq: (12052, 689)

----------------------------------------
LOADING DIAGNOSIS LABELS...
After diagnosis filter: 12052 admissions

------------------------------------

GAMENet

In [10]:
# ============================================================
# GAMENET (FAITHFUL + EARLY STOPPING FOR FAIR COMPARISON)
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from collections import defaultdict
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from sklearn.model_selection import KFold, train_test_split

device = "cuda" if torch.cuda.is_available() else "cpu"

# ============================================================
# HYPERPARAMS (paper-aligned)
# ============================================================
batch_size = 16
hidden_dim = 256
epochs = 50
lr = 1e-4              # original GAMENet scale
top_k = 20
lambda_ddi = 0.0       # fixed trade-off (paper style)

# ============================================================
# 1. DRUG DIMENSION + BINARY DDI
# ============================================================
n_drugs = ddi_severity_matrix.shape[0]

#  KEY FIX: binary DDI (paper)
ddi_binary = (ddi_severity_matrix > 0).astype(np.float32)
ddi_tensor = torch.tensor(ddi_binary, dtype=torch.float32, device=device)

# ============================================================
# 2. MULTI-HOT DRUG LABELS
# ============================================================
drug_labels_fixed = []

for seq in drug_labels_by_patient:
    seq_enc = np.zeros((len(seq), n_drugs), dtype=np.float32)
    for t, drugs in enumerate(seq):
        for d in drugs:
            if isinstance(d, (int, np.integer)) and d < n_drugs:
                seq_enc[t, d] = 1.0
    drug_labels_fixed.append(seq_enc)

# ============================================================
# 3. YOUR EMBEDDINGS (UNCHANGED)
# ============================================================
visits_all, drugs_all, patient_ids_all = [], [], []

for pid, lab, rad, drug_desc, drug_seq in zip(
    patient_ids,
    lab_sequences_by_patient,
    rad_sequences_by_patient,
    drug_desc_sequences_by_patient,
    drug_labels_fixed
):
    for t in range(len(lab)):
        emb = np.concatenate([lab[t], rad[t], drug_desc[t]])
        visits_all.append(emb)
        drugs_all.append(drug_seq[t])
        patient_ids_all.append(pid)

visits_all = np.array(visits_all)
drugs_all = np.array(drugs_all)

# Normalize
visits_all = (visits_all - visits_all.mean(0)) / (visits_all.std(0) + 1e-8)

# ============================================================
# 4. GROUP BY PATIENT
# ============================================================
p2v, p2d = defaultdict(list), defaultdict(list)

for v, d, pid in zip(visits_all, drugs_all, patient_ids_all):
    p2v[pid].append(v)
    p2d[pid].append(d)

sequences = [np.stack(v) for v in p2v.values()]
drug_sequences = [np.stack(v) for v in p2d.values()]

print(f"Number of patients: {len(sequences)}")
print(f"Average visits: {np.mean([len(s) for s in sequences]):.2f}")

# ============================================================
# 5. DATASET
# ============================================================
class VisitDataset(Dataset):
    def __init__(self, X, Y):
        self.X = [torch.tensor(x, dtype=torch.float32) for x in X]
        self.Y = [torch.tensor(y, dtype=torch.float32) for y in Y]

    def __len__(self): return len(self.X)

    def __getitem__(self, i):
        return self.X[i], self.Y[i]

def collate(batch):
    xs, ys = zip(*batch)
    lengths = torch.tensor([len(x) for x in xs])
    return pad_sequence(xs, batch_first=True), pad_sequence(ys, batch_first=True), lengths

# ============================================================
# 6. GRAPH BUILDING
# ============================================================
print("\nBuilding graphs...")

ehr_A = np.zeros((n_drugs, n_drugs), dtype=np.float32)

for seq in drug_sequences:
    for visit in seq:
        drugs = np.where(visit > 0)[0]
        for i in drugs:
            for j in drugs:
                if i != j:
                    ehr_A[i, j] += 1

def normalize(A):
    D = np.diag(1.0 / (np.sqrt(A.sum(1)) + 1e-8))
    return D @ A @ D

ehr_A = torch.tensor(normalize(ehr_A), dtype=torch.float32, device=device)
ddi_A = torch.tensor(normalize(ddi_binary), dtype=torch.float32, device=device)

print(f"EHR graph shape: {ehr_A.shape}")
print(f"DDI graph shape: {ddi_A.shape}")

# ============================================================
# 7. TRUE GAMENET MODEL
# ============================================================
class GAMENet(nn.Module):
    def __init__(self, input_dim, n_drugs, hidden_dim):
        super().__init__()

        self.input_proj = nn.Linear(input_dim, hidden_dim)
        self.gru = nn.GRU(hidden_dim, hidden_dim, batch_first=True)

        self.drug_emb = nn.Parameter(torch.randn(n_drugs, hidden_dim) / np.sqrt(hidden_dim))

        self.gcn_ehr = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.gcn_ddi = nn.Linear(hidden_dim, hidden_dim, bias=False)

        self.alpha = nn.Parameter(torch.tensor(0.5))

        self.out = nn.Linear(hidden_dim * 3, n_drugs)

    def graph_memory(self):
        H = self.drug_emb
        ehr = torch.relu(self.gcn_ehr(ehr_A @ H))
        ddi = torch.relu(self.gcn_ddi(ddi_A @ H))
        return torch.sigmoid(self.alpha) * ehr - (1 - torch.sigmoid(self.alpha)) * ddi

    def attention(self, q, k, v):
        attn = torch.softmax(q @ k.T, dim=-1)
        return attn @ v

    def forward(self, x, lengths):
        B, T, _ = x.size()

        x = torch.relu(self.input_proj(x))
        packed = nn.utils.rnn.pack_padded_sequence(
            x, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        out, _ = self.gru(packed)
        h, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True, total_length=T)

        M_g = self.graph_memory()
        outputs = torch.zeros(B, T, n_drugs, device=x.device)

        for b in range(B):
            mem_k, mem_v = [], []
            for t in range(lengths[b]):
                h_t = h[b, t]

                if t == 0:
                    o_d = torch.zeros_like(h_t)
                else:
                    k = torch.stack(mem_k)
                    v = torch.stack(mem_v)
                    o_d = self.attention(h_t, k, v)
                    o_d = o_d @ self.drug_emb

                o_g = self.attention(h_t, M_g, M_g)

                logits = self.out(torch.cat([h_t, o_g, o_d], dim=-1))
                outputs[b, t] = logits

                prob = torch.sigmoid(logits).detach()
                mem_k.append(h_t.detach())
                mem_v.append(prob)

        return outputs

# ============================================================
# 8. TRUE LOSS (NO L1, NO pos_weight)
# ============================================================
def loss_fn(logits, targets, lengths):
    B, T, D = logits.shape

    mask = torch.zeros(B, T, 1, device=device)
    for i in range(B):
        mask[i, :lengths[i]] = 1

    # BCE (plain)
    bce = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
    bce = (bce * mask).sum() / mask.sum()

    # Binary DDI loss
    probs = torch.sigmoid(logits)
    ddi_loss = 0
    count = 0

    for i in range(B):
        for t in range(lengths[i]):
            p = probs[i, t]
            ddi_loss += (torch.outer(p, p) * ddi_tensor).sum()
            count += 1

    ddi_loss = ddi_loss / (count + 1e-8)

    return bce + lambda_ddi * ddi_loss, bce.item(), ddi_loss.item()

# ============================================================
# 9. CROSS VALIDATION (UNCHANGED + EARLY STOPPING)
# ============================================================
print("\n" + "="*60)
print("Starting 10-Fold Cross Validation - GAMENet")
print("="*60)

kf = KFold(n_splits=10, shuffle=True, random_state=42)

all_recall = []
all_ndcg = []
all_ddi_rate = []
all_ddi_severity = []

for fold, (tr, te) in enumerate(kf.split(sequences), 1):

    print(f"\n{'='*60}")
    print(f"Fold {fold}/10")
    print(f"{'='*60}")

    train_idx, val_idx = train_test_split(tr, test_size=0.2, random_state=42)

    train_ds = VisitDataset([sequences[i] for i in train_idx],
                            [drug_sequences[i] for i in train_idx])
    val_ds = VisitDataset([sequences[i] for i in val_idx],
                          [drug_sequences[i] for i in val_idx])
    test_ds = VisitDataset([sequences[i] for i in te],
                           [drug_sequences[i] for i in te])

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, collate_fn=collate)
    val_loader = DataLoader(val_ds, batch_size=batch_size, collate_fn=collate)
    test_loader = DataLoader(test_ds, batch_size=batch_size, collate_fn=collate)

    model = GAMENet(sequences[0].shape[1], n_drugs, hidden_dim).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    best_val_loss = float('inf')
    patience = 0

    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for x, y, lengths in train_loader:
            x, y, lengths = x.to(device), y.to(device), lengths.to(device)

            optimizer.zero_grad()
            logits = model(x, lengths)
            loss, _, _ = loss_fn(logits, y, lengths)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)

        # validation
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for x, y, lengths in val_loader:
                x, y, lengths = x.to(device), y.to(device), lengths.to(device)
                loss, _, _ = loss_fn(model(x, lengths), y, lengths)
                val_loss += loss.item()
        val_loss /= len(val_loader)

        print(f"Epoch {epoch+1}/{epochs} | Loss: {avg_loss:.4f} | Val: {val_loss:.4f}")

        # early stopping (kept for fairness)
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience = 0
            torch.save(model.state_dict(), f'gamenet_best_model_fold{fold}.pt')
        else:
            patience += 1
            if patience >= 7:
                print(f"Early stopping at epoch {epoch+1}")
                break

    model.load_state_dict(torch.load(f'gamenet_best_model_fold{fold}.pt', weights_only=False))
    model.eval()

    # ================== EVALUATION (UNCHANGED) ==================
    y_true_drug, y_score_drug = [], []

    with torch.no_grad():
        for x, y, lengths in test_loader:
            x, y, lengths = x.to(device), y.to(device), lengths.to(device)
            logits = model(x, lengths)

            for i in range(len(lengths)):
                L = lengths[i].item()
                if L <= 1: continue
                y_true_drug.append(y[i, 1:L].cpu().numpy())
                y_score_drug.append(torch.sigmoid(logits[i, 1:L]).cpu().numpy())

    y_true_drug_arr = np.vstack(y_true_drug) if y_true_drug else np.array([])
    y_score_drug_arr = np.vstack(y_score_drug) if y_score_drug else np.array([])

    if len(y_true_drug_arr) > 0:
        recall_val = float(np.mean(recall_at_k(y_true_drug_arr, y_score_drug_arr, top_k)))
        ndcg_val = float(ndcg_at_k(y_true_drug_arr, y_score_drug_arr, top_k))
        ddi_rate_val, ddi_sev_avg = ddi_rate(y_score_drug_arr, ddi_binary, top_k)
    else:
        recall_val = ndcg_val = ddi_rate_val = ddi_sev_avg = 0.0

    all_recall.append(recall_val)
    all_ndcg.append(ndcg_val)
    all_ddi_rate.append(ddi_rate_val)
    all_ddi_severity.append(ddi_sev_avg)

    print(f"\nFold {fold} Results:")
    print(f"  Drugs - Recall@{top_k}: {recall_val:.4f} | NDCG@{top_k}: {ndcg_val:.4f}")
    print(f"  Safety - DDI rate: {ddi_rate_val:.4f} | Avg severity: {ddi_sev_avg:.4f}")

# ============================================================
# SUMMARY
# ============================================================
print("\n" + "="*60)
print("10-Fold CV Summary - GAMENet")
print("="*60)
print(f"Recall@{top_k}   : {np.mean(all_recall):.4f} ± {np.std(all_recall):.4f}")
print(f"NDCG@{top_k}     : {np.mean(all_ndcg):.4f} ± {np.std(all_ndcg):.4f}")
print(f"DDI rate         : {np.mean(all_ddi_rate):.4f} ± {np.std(all_ddi_rate):.4f}")
print(f"Avg DDI severity : {np.mean(all_ddi_severity):.4f} ± {np.std(all_ddi_severity):.4f}")

Number of patients: 7011
Average visits: 1.72

Building graphs...
EHR graph shape: torch.Size([2374, 2374])
DDI graph shape: torch.Size([2374, 2374])

Starting 10-Fold Cross Validation - GAMENet

Fold 1/10
Epoch 1/50 | Loss: 309.8630 | Val: 84.2159
Epoch 2/50 | Loss: 80.5016 | Val: 79.1048
Epoch 3/50 | Loss: 78.6527 | Val: 78.6308
Epoch 4/50 | Loss: 78.3840 | Val: 78.5306
Epoch 5/50 | Loss: 78.3215 | Val: 78.4845
Epoch 6/50 | Loss: 78.2147 | Val: 78.4326
Epoch 7/50 | Loss: 78.1067 | Val: 78.4702
Epoch 8/50 | Loss: 78.1704 | Val: 78.4627
Epoch 9/50 | Loss: 78.1550 | Val: 78.4769
Epoch 10/50 | Loss: 78.0571 | Val: 78.4812
Epoch 11/50 | Loss: 78.0185 | Val: 78.5141
Epoch 12/50 | Loss: 77.9530 | Val: 78.6021
Epoch 13/50 | Loss: 78.0185 | Val: 78.5280
Early stopping at epoch 13

Fold 1 Results:
  Drugs - Recall@20: 0.3757 | NDCG@20: 0.6020
  Safety - DDI rate: 0.0526 | Avg severity: 1.0000

Fold 2/10
Epoch 1/50 | Loss: 311.9788 | Val: 83.2339
Epoch 2/50 | Loss: 79.8118 | Val: 78.6362
Epoch 